### 目的: 將Longitudinal_AD_張醫師_20250921_V3.xlsx檔案整併成 張瓊之_給馬老師資料20250925_V4 格式，GPT建議新增兩張表: 病人層級資料表patient_level_summary CDR event 層級資料表CDR_event_with_blood

In [ ]:
import pandas as pd 
import numpy as np 
import sys

sys.path.append("../data/raw")

In [ ]:
file_path = "../data/raw/Longitudinal_AD_張醫師_20250921_V3.xlsx"

demo = pd.read_excel(file_path, sheet_name="demographic")
onset = pd.read_excel(file_path, sheet_name="發病年紀")
mmse = pd.read_excel(file_path, sheet_name="MMSE longitudinal")
blood = pd.read_excel(file_path, sheet_name="blood_data_20250921")
casi = pd.read_excel(file_path, sheet_name="CASI longitudinal")

### 整理demo

In [ ]:
demo.head()

In [ ]:
# 用第 0 列當英文欄名
demo.columns = demo.iloc[0]

# 刪掉第 0 列
demo = demo.iloc[1:].copy()

# 統一病人 ID 欄位名稱
demo = demo.rename(columns={"Count Number": "patientID"})

# 型別整理
demo["patientID"] = demo["patientID"].astype(int)

In [ ]:
demo.head()

### 整理onset

In [ ]:
onset = onset.rename(columns={
    "Count Number": "patientID",
    "age of onset": "age_of_onset",
    "Biological category": "Biological category_onset"
})

onset["patientID"] = onset["patientID"].astype(int)
onset.head()

### 整理MMSE longitudinal

In [ ]:
mmse = mmse.rename(columns={
    "Count Number": "patientID",
    "MMSE(numerical)": "MMSE",
    "CDR (scale)": "CDR",
    "CDR-SOB(numerical)": "CDR_SOB"
})

mmse["patientID"] = mmse["patientID"].astype(int)
mmse["Date"] = pd.to_datetime(mmse["Date"])
mmse = mmse.sort_values(["patientID", "Date"]).reset_index(drop=True)

mmse.head()

In [ ]:
mmse["MMSE_repeat_no"] = mmse.groupby("patientID").cumcount() + 1
mmse["CDR_repeat_no"] = mmse.groupby("patientID").cumcount() + 1

In [ ]:
mmse_valid = mmse.dropna(subset=["MMSE"]).copy()

mmse_first = (
    mmse_valid
    .sort_values(["patientID", "Date"])
    .groupby("patientID")
    .first()
    .reset_index()
)

mmse_final = (
    mmse_valid
    .sort_values(["patientID", "Date"])
    .groupby("patientID")
    .last()
    .reset_index()
)

In [ ]:
mmse_summary = mmse_first[[
    "patientID", "Date", "MMSE"
]].rename(columns={
    "Date": "MMSE Date_first",
    "MMSE": "MMSE_first"
})

mmse_summary = mmse_summary.merge(
    mmse_final[[
        "patientID", "Date", "MMSE", "MMSE_repeat_no"
    ]].rename(columns={
        "Date": "MMSE Date_final",
        "MMSE": "MMSE_final",
        "MMSE_repeat_no": "MMSE Repeat no._final"
    }),
    on="patientID",
    how="left"
)

In [ ]:
mmse_summary.head()

## 產生 CDR 病人層級摘要

In [ ]:
cdr_valid = mmse.dropna(subset=["CDR"]).copy()
cdr_valid = cdr_valid.sort_values(["patientID", "Date"])

cdr_valid["CDR_repeat_no"] = cdr_valid.groupby("patientID").cumcount() + 1

In [ ]:
cdr_first = (
    cdr_valid
    .groupby("patientID")
    .nth(0)
    .reset_index()
)

cdr_next = (
    cdr_valid
    .groupby("patientID")
    .nth(1)
    .reset_index()
)

cdr_summary = cdr_first[[
    "patientID", "CDR_repeat_no", "Date", "CDR"
]].rename(columns={
    "CDR_repeat_no": "CDR Repeat_first",
    "Date": "CDR Date_first",
    "CDR": "CDR_first"
})

cdr_summary = cdr_summary.merge(
    cdr_next[[
        "patientID", "CDR_repeat_no", "Date", "CDR"
    ]].rename(columns={
        "CDR_repeat_no": "CDR Repeat_next",
        "Date": "CDR Date_next",
        "CDR": "CDR_next"
    }),
    on="patientID",
    how="left"
)

In [ ]:
# =========================
# CDR summary
# 有 CDR 由小變大：抓第一次變大的前後兩筆
# 沒有 CDR 由小變大：抓最後兩筆
# =========================

cdr_valid = mmse.dropna(subset=["CDR"]).copy()

cdr_valid = (
    cdr_valid
    .sort_values(["patientID", "Date"])
    .reset_index(drop=True)
)

# 每位病人的第幾次 CDR 紀錄
cdr_valid["CDR_repeat_no"] = (
    cdr_valid.groupby("patientID").cumcount() + 1
)

# 建立下一筆 CDR 資訊
cdr_valid["CDR_next_tmp"] = cdr_valid.groupby("patientID")["CDR"].shift(-1)
cdr_valid["Date_next_tmp"] = cdr_valid.groupby("patientID")["Date"].shift(-1)
cdr_valid["CDR_repeat_no_next_tmp"] = (
    cdr_valid.groupby("patientID")["CDR_repeat_no"].shift(-1)
)

# 判斷這一筆到下一筆是否 CDR 變大
cdr_valid["event_pair"] = cdr_valid["CDR_next_tmp"] > cdr_valid["CDR"]

# =========================
# 1. 有 event 的病人：抓第一次 CDR 變大的 pair
# =========================

cdr_event = (
    cdr_valid[cdr_valid["event_pair"]]
    .sort_values(["patientID", "Date"])
    .groupby("patientID")
    .first()
    .reset_index()
)

cdr_event_summary = cdr_event[[
    "patientID",
    "CDR_repeat_no",
    "Date",
    "CDR",
    "CDR_repeat_no_next_tmp",
    "Date_next_tmp",
    "CDR_next_tmp"
]].rename(columns={
    "CDR_repeat_no": "CDR Repeat_first",
    "Date": "CDR Date_first",
    "CDR": "CDR_first",
    "CDR_repeat_no_next_tmp": "CDR Repeat_next",
    "Date_next_tmp": "CDR Date_next",
    "CDR_next_tmp": "CDR_next"
})

cdr_event_summary["event"] = 1

# =========================
# 2. 沒有 event 的病人：抓最後兩筆 CDR
# =========================

event_patient_ids = set(cdr_event_summary["patientID"])

cdr_no_event = cdr_valid[
    ~cdr_valid["patientID"].isin(event_patient_ids)
].copy()


def get_last_two_cdr(group):
    group = group.sort_values("Date").copy()

    # 有至少兩筆 CDR：抓最後兩筆
    if len(group) >= 2:
        first = group.iloc[-2]
        next_ = group.iloc[-1]

        return pd.Series({
            "CDR Repeat_first": first["CDR_repeat_no"],
            "CDR Date_first": first["Date"],
            "CDR_first": first["CDR"],
            "CDR Repeat_next": next_["CDR_repeat_no"],
            "CDR Date_next": next_["Date"],
            "CDR_next": next_["CDR"],
            "event": 0
        })

    # 只有一筆 CDR：沒有 next，無法比較
    else:
        first = group.iloc[-1]

        return pd.Series({
            "CDR Repeat_first": first["CDR_repeat_no"],
            "CDR Date_first": first["Date"],
            "CDR_first": first["CDR"],
            "CDR Repeat_next": np.nan,
            "CDR Date_next": pd.NaT,
            "CDR_next": np.nan,
            "event": np.nan
        })


cdr_no_event_summary = (
    cdr_no_event
    .groupby("patientID")
    .apply(get_last_two_cdr)
    .reset_index()
)

# =========================
# 3. 合併 event / no-event
# =========================

cdr_summary = pd.concat(
    [cdr_event_summary, cdr_no_event_summary],
    ignore_index=True
)

cdr_summary = (
    cdr_summary
    .sort_values("patientID")
    .reset_index(drop=True)
)

### 計算 CDR event 

In [ ]:
#CDR_next > CDR_first，event = 1 否則 event = 0
cdr_summary["event"] = np.where(
    cdr_summary["CDR_next"] > cdr_summary["CDR_first"],
    1,
    0
)

# 如果 CDR_next 是缺失，表示只有一次 CDR 紀錄，設為 missing
cdr_summary.loc[cdr_summary["CDR_next"].isna(), "event"] = np.nan

### 合併成病人層級資料表

In [ ]:
patient_level = (
    demo
    .merge(onset[["patientID", "age_of_onset"]], on="patientID", how="left")
    .merge(mmse_summary, on="patientID", how="left")
    .merge(cdr_summary, on="patientID", how="left")
)

patient_level.head()

In [ ]:
patient_level["patientID"].duplicated().sum()

In [ ]:
cols = [
    "patientID",
    "Diagnosis group at enrollment",
    "Biological category",
    "Apoe4 status",
    "Gender",
    "Age at first MMSE",
    "Education",
    "HT",
    "DM",
    "Hyperlipidemia",
    "age_of_onset",
    "MMSE Date_first",
    "MMSE_first",
    "MMSE Repeat no._final",
    "MMSE Date_final",
    "MMSE_final",
    "followup_years",
    "MMSE_decline",
    "MMSE_decline_per_year",
    "Fast_decline",
    "CDR Repeat_first",
    "CDR Date_first",
    "CDR_first",
    "CDR Repeat_next",
    "CDR Date_next",
    "CDR_next",
    "event"
]

patient_level = patient_level[[c for c in cols if c in patient_level.columns]]

In [ ]:
patient_level.head()

In [ ]:
patient_level.to_excel("../data/processed/patient_level_summary.xlsx", index=False)

## 建立CDR event 層級資料表

In [ ]:
import pandas as pd
import numpy as np

file_path = "../data/raw/Longitudinal_AD_張醫師_20250921_V3.xlsx"

mmse = pd.read_excel(file_path, sheet_name="MMSE longitudinal")
blood = pd.read_excel(file_path, sheet_name="blood_data_20250921")

In [ ]:
mmse = mmse.rename(columns={
    "Count Number": "patientID",
    "MMSE(numerical)": "MMSE",
    "CDR (scale)": "CDR",
    "CDR_M (scale)": "CDR_M",
    "CDR_O (scale)": "CDR_O",
    "CDR_J (scale)": "CDR_J",
    "CDR_C (scale)": "CDR_C",
    "CDR_H (scale)": "CDR_H",
    "CDR_P (scale)": "CDR_P",
    "CDR-SOB(numerical)": "CDR_SOB"
})

mmse["patientID"] = mmse["patientID"].astype(int)
mmse["Date"] = pd.to_datetime(mmse["Date"])

mmse = mmse.sort_values(["patientID", "Date"]).reset_index(drop=True)

### 建立 CDR 前後變化 如果這次 CDR > 上一次 CDR，表示 CDR 惡化，event = 1 如果沒有惡化，event = 0

In [ ]:
# 先只保留有 CDR 的列
cdr = mmse.dropna(subset=["CDR"]).copy()

cdr = cdr.sort_values(["patientID", "Date"]).reset_index(drop=True)
cdr["CDR_repeat_no"] = cdr.groupby("patientID").cumcount() + 1

In [ ]:
# 計算上一筆 CDR 與上一筆日期

cdr["CDR_previous"] = cdr.groupby("patientID")["CDR"].shift(1)
cdr["Date_previous"] = cdr.groupby("patientID")["Date"].shift(1)
cdr["CDR_repeat_previous"] = cdr.groupby("patientID")["CDR_repeat_no"].shift(1)

In [ ]:
# 判斷是否出現 CDR event：
cdr["是否出現event"] = np.where(
    cdr["CDR"] > cdr["CDR_previous"],
    1,
    0
)

# 第一筆沒有 previous，不能判斷 event，設為 0
cdr.loc[cdr["CDR_previous"].isna(), "是否出現event"] = 0

### 抓每位病人的第一次 CDR event

In [ ]:
cdr_events = cdr[cdr["是否出現event"] == 1].copy()

first_cdr_event = (
    cdr_events
    .sort_values(["patientID", "Date"])
    .groupby("patientID")
    .first()
    .reset_index()
)

cdr_event_table = first_cdr_event[[
    "patientID",
    "Date",
    "CDR_previous",
    "CDR",
    "CDR_repeat_previous",
    "CDR_repeat_no",
    "是否出現event"
]].rename(columns={
    "Date": "date_of_CDR_change",
    "CDR_previous": "CDR_before",
    "CDR": "CDR_after",
    "CDR_repeat_previous": "CDR_repeat_before",
    "CDR_repeat_no": "CDR_repeat_after"
})

### 保流沒有 event 的病人

In [ ]:
all_patients = pd.DataFrame({
    "patientID": sorted(cdr["patientID"].unique())
})

cdr_event_table = all_patients.merge(
    cdr_event_table,
    on="patientID",
    how="left"
)

cdr_event_table["是否出現event"] = cdr_event_table["是否出現event"].fillna(0).astype(int)

### 整理blood data

In [ ]:
blood = blood.rename(columns={
    "Count": "patientID"
})

blood["patientID"] = blood["patientID"].astype(int)
blood["Date"] = pd.to_datetime(blood["Date"])

In [ ]:
#把 -1 轉成 missing
blood_value_cols = [
    "HDL-C",
    "VLDL-C",
    "LDL-C",
    "T-Cholesterol",
    "Triglyceride",
    "AC sugar level",
    "HbA1c"
]

blood[blood_value_cols] = blood[blood_value_cols].replace(-1, np.nan)

In [ ]:
# 對每個有 CDR event 的病人，找同一位病人中，血液檢查日期最接近 date_of_CDR_change 的一筆
event_positive = cdr_event_table[
    cdr_event_table["是否出現event"] == 1
].copy()

event_positive = event_positive[[
    "patientID",
    "date_of_CDR_change"
]]

In [ ]:
# 先和 blood data 合併，讓每個 event 對應該病人的所有 blood records
event_blood_candidates = event_positive.merge(
    blood,
    on="patientID",
    how="left"
)

# 計算 event 日期和抽血日期差距
event_blood_candidates["days_diff"] = (
    event_blood_candidates["Date"] - event_blood_candidates["date_of_CDR_change"]
).abs().dt.days

# 每位病人取差距最小的 blood record
closest_blood = (
    event_blood_candidates
    .sort_values(["patientID", "days_diff"])
    .groupby("patientID")
    .first()
    .reset_index()
)

In [ ]:
# 整理欄位
closest_blood = closest_blood.rename(columns={
    "Date": "blood_date_closest_to_CDR_change"
})

closest_blood = closest_blood[[
    "patientID",
    "blood_date_closest_to_CDR_change",
    "HDL-C",
    "VLDL-C",
    "LDL-C",
    "T-Cholesterol",
    "Triglyceride",
    "AC sugar level",
    "HbA1c",
    "days_diff"
]]

In [ ]:
# 把最近血液資料接回 CDR event table
cdr_event_with_blood = cdr_event_table.merge(
    closest_blood,
    on="patientID",
    how="left"
)

final_cols = [
    "patientID",
    "date_of_CDR_change",
    "blood_date_closest_to_CDR_change",
    "CDR_before",
    "CDR_after",
    "CDR_repeat_before",
    "CDR_repeat_after",
    "HDL-C",
    "VLDL-C",
    "LDL-C",
    "T-Cholesterol",
    "Triglyceride",
    "AC sugar level",
    "HbA1c",
    "days_diff",
    "是否出現event"
]

cdr_event_with_blood = cdr_event_with_blood[
    [c for c in final_cols if c in cdr_event_with_blood.columns]
]

In [ ]:
cdr_event_with_blood.head()

In [ ]:
cdr_event_with_blood.to_excel(
    "../data/processed/CDR_event_with_blood.xlsx",
    index=False
)